# Random Forest



## Creamos el pipeline del modelo

El dataset esta mezclado en train, val y test ya que val lo vamos ausar en deep learnin para hace ealry stopping. Pero en shallow learning para comaprar modelos con resutlados mas preresentativos dle rendimeitno combiene mas usar cross validation. 

Para usar cross validation, tenemos que juntar train+val. 

Aqui surge un problema, y es que anterioremtne en el notebook 2 hicimos fit transform del vectorizer en tain y transform en val lo cual para usarlos de forma serada esta bein y no causa probelmas pero al jutnarlos para hacer crossv laidation es posible que en algun fold haya un validation set de parte del train con el que s ehaya hecho fit, metiendo data leakage. 

Para ello usaremos un pipeline y vovleremos a vectorizar. 

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_validate
import pandas as pd

# 1. Definir los modelos
models = {
    "Naive Bayes": Pipeline([
        ("vec", build_vectorizer(min_df=2)),
        ("clf", MultinomialNB())
    ]),

    "LinearSVC": Pipeline([
        ("vec", build_vectorizer(min_df=2)),
        ("clf", LinearSVC(class_weight="balanced"))
    ])
}

# 2. Misma Cross-Validation para ambos
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3. Evaluar todos los modelos
results = []

for name, pipe in models.items():

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring = ["f1", "accuracy","precision","recall","matthews_corrcoef"],
        n_jobs=-1
    )

    results.append({
        "Modelo": name,
        "F1": scores["test_f1"].mean(),
        "F1_std": scores["test_f1"].std(),
        "Accuracy": scores["test_accuracy"].mean(),
        "Accuracy_std": scores["test_accuracy"].std(),
        "Precision": scores["test_precision"].mean(),
        "Precision_std": scores["test_precision"].std(),
        "Recall": scores["test_recall"].mean(),
        "Recall_std": scores["test_recall"].std(),
        "MCC": scores["test_matthews_corrcoef"].mean(),
        "MCC_std": scores["test_matthews_corrcoef"].std(),

    })


# 4. Comparar resultados
df_results = pd.DataFrame(results)
print(df_results.sort_values("F1", ascending=False))

        Modelo        F1    F1_std  Accuracy  Accuracy_std  Precision  \
1    LinearSVC  0.638743  0.027505  0.937035      0.004639   0.651793   
0  Naive Bayes  0.006910  0.005642  0.910670      0.000831   0.466667   

   Precision_std    Recall  Recall_std       MCC   MCC_std  
1       0.030237  0.627445    0.037073  0.604795  0.029667  
0       0.452155  0.003488    0.002848  0.031043  0.038022  


 resutlaods optuna: 
 Mejor MCC: 0.6128893579281603

 Mejores parámetros: {'C': 1.9892802274290187, 'min_df': 1}

In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Definimos los folds una sola vez
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def objective(trial):

    # Hiperparámetros propuestos por Optuna
    C = trial.suggest_float("C", 1e-3, 100, log=True)
    min_df = trial.suggest_int("min_df", 1, 5)

    pipe = Pipeline([
        ("vec", build_vectorizer(min_df=min_df)),
        ("clf", LinearSVC(
            C=C,
            class_weight="balanced",
            random_state=42
        ))
    ])

    # Cross-Validation automática
    scores = cross_val_score(
        estimator=pipe,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring="matthews_corrcoef", #optimizaoms MCC la metrica mas robusto y que deidfne el rendimeinto real 
        n_jobs=-1
    )

    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Mejor Mcc:", study.best_value)
print("Mejores parámetros:", study.best_params)

[I 2026-11-10 10:55:45,624] A new study created in memory with name: no-name-e8314556-a69e-4e79-8cbd-ca6eccedf075
[I 2026-11-10 10:56:02,871] Trial 0 finished with value: 0.539975819330101 and parameters: {'C': 25.81750804627779, 'min_df': 4}. Best is trial 0 with value: 0.539975819330101.
[I 2026-11-10 10:56:16,533] Trial 1 finished with value: 0.5500260819635188 and parameters: {'C': 0.026812269573351692, 'min_df': 1}. Best is trial 1 with value: 0.5500260819635188.
[I 2026-11-10 10:56:26,657] Trial 2 finished with value: 0.4781551222893512 and parameters: {'C': 0.001152680392082014, 'min_df': 3}. Best is trial 1 with value: 0.5500260819635188.
[I 2026-11-10 10:56:34,737] Trial 3 finished with value: 0.540995420904473 and parameters: {'C': 0.022583168177875514, 'min_df': 3}. Best is trial 1 with value: 0.5500260819635188.
[I 2026-11-10 10:56:43,783] Trial 4 finished with value: 0.5833925955186372 and parameters: {'C': 0.07872609447403862, 'min_df': 2}. Best is trial 4 with value: 0.5

Mejor F1: 0.6128893579281603
Mejores parámetros: {'C': 1.9892802274290187, 'min_df': 1}
